# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates the use of the [`mlcroissant`](https://github.com/mlcommons/croissant) library to load, explore, and process data and metadata from the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa), which details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples from mass spectrometry experiments.

### Dataset Source
The dataset source is accessible via a Croissant schema URL:

- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We'll begin by importing the necessary libraries and loading the Croissant metadata. The dataset metadata provides dataset-level description and available record sets/fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Let's explore which record sets, fields, and columns are available in this dataset, referencing them by their `@id` identifiers as specified by the Croissant specification.

Below, we will print each record set's `@id`, its field names and corresponding `@id`s, and provide an example record for each.

In [ ]:
# List all available record sets and their field @ids
for record_set in metadata.record_sets:
    print(f"RecordSet name: {record_set.name}")
    print(f"RecordSet @id: {record_set.id}")
    print("Fields:")
    for field in record_set.fields:
        print(f"    Field name: {field.name} | @id: {field.id}")
    print("\nExample record:")
    iterator = dataset.records(record_set=record_set.id)
    try:
        print(next(iterator))
    except StopIteration:
        print("   No records found.")
    print("---\n")

## 3. Data Extraction

Let's load the data from one or more record sets into pandas DataFrames for further processing. *All* entity references are handled by their `@id` as per best practice.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

if record_set_ids:
    print('RecordSet IDs:', record_set_ids)

    # Preview the columns of the first record set with records
    main_record_set_id = next((rs_id for rs_id in record_set_ids if rs_id in dataframes), None)
    if main_record_set_id:
        print(f"Columns in RecordSet '{main_record_set_id}':\n", dataframes[main_record_set_id].columns.tolist())
        dataframes[main_record_set_id].head()
    else:
        print("No record sets contain records.")
else:
    print("No record sets available.")

## 4. Exploratory Data Analysis (EDA)

We will now select a numeric field from the main record set and demonstrate filtering, normalization, and grouping operations. All fields are referenced by their `@id`, and all operations are performed on the loaded DataFrame.

In [ ]:
# For this dataset, we typically have abundance or MW (Molecular Weight) columns identified by their field @id.
# Let's auto-detect a numeric field (float or int) for demonstration.

main_rs_id = main_record_set_id  # From previous cell

df = dataframes[main_rs_id]

# Try to identify a numeric field based on datatype in the metadata
record_set_obj = next(rs for rs in metadata.record_sets if rs.id == main_rs_id)
numeric_field_id = None
group_field_id = None
for field in record_set_obj.fields:
    # Prefer a field explicitly typed as Float or Integer
    if field.data_type in ('Float', 'Integer', 'Number'):
        if numeric_field_id is None:
            numeric_field_id = field.id
        # Look for a second field for group demonstration
    else:
        if group_field_id is None:  # Pick a non-numeric field for grouping
            group_field_id = field.id

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}\n")

# Coerce to numeric in case any string parsing is required
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].quantile(0.8) if df[numeric_field_id].notnull().any() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric column
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field and show mean of numeric field
if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and how it varies by the grouping field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Distribution
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If we have a group field, show boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        top_groups = df[group_field_id].value_counts().head(6).index
        subdf = df[df[group_field_id].isin(top_groups)]
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=subdf)
        plt.title(f"{numeric_field_id} by {group_field_id} (top categories)")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We loaded the mass spectrometry dataset using its Croissant schema and the `mlcroissant` library, exploring the dataset structure via its entity `@id` identifiers.
- We extracted all records and performed common data filtering and normalization tasks on a numeric field, and visualized key distributions and groupings.

This process enables reproducible, transparent, and robust data loading and analysis, closely tied to standardized metadata. For further exploration, consult the dataset's [FAIR^2 entry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) or the [Croissant specification](https://mlcommons.org/croissant/spec/).